## Постановка задачи
Необходимо рассчитать следующие метрики:
- leave to start ratio - коэффициент соотношения покинувших пользовтелей сайт к тем, кто начал участие
- days to churn - время с последнего действия на сайте до момента ухода

## Данные 
Данные представлены с трех независимых источников и состоят из следующих признаков:
- Unnamed: 0 - ID строки
- topic_id - идентификационный номер топика. Топик состоит из одного вопроса и всех ответов  к нему
- id — идентификационный номер поста
- parent_id — идентификационный номер родительского поста для ответов или -1 для вопросов
- post_type — тип поста. Посты бывают только двух типов: вопрос — 1, ответ — 2.
- created_at — дата создания публикации
- author_id — идентификационный номер автора поста
- post_count - количество постов в вопросе
- view_count - количество просмотров
- reply_time - время ответа
- is_accepted_answer - ответ на вопрос?
- author_username - имя пользователя

## Как определять последнее действие пользователя?

Способ 1 - Последнее действие выбирается исходя из даты последнего зарегистрированного в БД

Способ 2 - Использовать модель Customer Livetime Value, через вероятность.

Загружаем библиотеки

In [165]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
from pymc_marketing.clv import ParetoNBDModel
from pymc_marketing.clv.utils import rfm_summary
import seaborn as sns
import arviz as az
import os


sns.set(style="darkgrid")

Загружаем датасет и смотрим на признаки

In [166]:
elastic = pd.read_csv(r'..\data\raw_data\elastic.csv')
freecodecamp = pd.read_csv(r'..\data\raw_data\freecodecamp.csv')
pythoncom = pd.read_csv(r'..\data\raw_data\python.csv')
datas = {
    'elastic': elastic,
    'freecodecamp': freecodecamp,
    'python': pythoncom
}

In [167]:
elastic[elastic['author_id'] <= -2].head()

,Unnamed: 0,topic_id,id,parent_id,post_type,posts_count,view_count,created_at,reply_time,is_accepted_answer,author_id,author_username
232546,232546,346432,1333928,1333871,2,0,0,2023-11-04 21:13:31.488000+00:00,0.4,False,-2,discobot


In [168]:
freecodecamp[freecodecamp['author_id'] <= -0].head()

,Unnamed: 0,topic_id,id,parent_id,post_type,posts_count,view_count,created_at,reply_time,is_accepted_answer,author_id,author_username
4,4,636351,1882964,1760497,2,0,0,2024-03-09 01:04:07.581000+00:00,182.53,False,-1,system
9,9,636343,1882965,1760483,2,0,0,2024-03-09 01:07:07.352000+00:00,182.56,False,-1,system
13,13,636329,1882967,1760452,2,0,0,2024-03-09 01:11:06.063000+00:00,182.60,False,-1,system
18,18,636366,1882971,1760537,2,0,0,2024-03-09 01:19:07.071000+00:00,182.51,False,-1,system
24,24,636331,1882977,1760455,2,0,0,2024-03-09 01:23:03.098000+00:00,182.60,False,-1,system


In [169]:
pythoncom[pythoncom['author_id'] <= -1].head()

,Unnamed: 0,topic_id,id,parent_id,post_type,posts_count,view_count,created_at,reply_time,is_accepted_answer,author_id,author_username
1828,1828,18844,114046,66593,2,0,0,2023-09-07 13:00:06.402000+00:00,365.0,False,-1,system
2253,2253,18988,114958,67116,2,0,0,2023-09-12 11:22:34.476000+00:00,365.0,False,-1,system
2658,2658,21087,127460,74589,2,0,0,2023-11-15 12:51:01.256000+00:00,365.0,False,-1,system
3062,3062,19599,118528,69309,2,0,0,2023-09-30 17:54:43.405000+00:00,365.0,False,-1,system
4088,4088,24059,147657,86468,2,0,0,2024-02-21 00:20:53.997000+00:00,365.0,False,-1,system


In [170]:
elastic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366549 entries, 0 to 366548
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          366549 non-null  int64  
 1   topic_id            366549 non-null  int64  
 2   id                  366549 non-null  int64  
 3   parent_id           366549 non-null  int64  
 4   post_type           366549 non-null  int64  
 5   posts_count         366549 non-null  int64  
 6   view_count          366549 non-null  int64  
 7   created_at          366549 non-null  object 
 8   reply_time          366549 non-null  float64
 9   is_accepted_answer  366549 non-null  bool   
 10  author_id           366549 non-null  int64  
 11  author_username     366549 non-null  object 
dtypes: bool(1), float64(1), int64(8), object(2)
memory usage: 31.1+ MB


In [171]:
freecodecamp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351988 entries, 0 to 351987
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          351988 non-null  int64  
 1   topic_id            351988 non-null  int64  
 2   id                  351988 non-null  int64  
 3   parent_id           351988 non-null  int64  
 4   post_type           351988 non-null  int64  
 5   posts_count         351988 non-null  int64  
 6   view_count          351988 non-null  int64  
 7   created_at          351988 non-null  object 
 8   reply_time          351988 non-null  float64
 9   is_accepted_answer  351988 non-null  bool   
 10  author_id           351988 non-null  int64  
 11  author_username     351988 non-null  object 
dtypes: bool(1), float64(1), int64(8), object(2)
memory usage: 29.9+ MB


In [172]:
pythoncom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135409 entries, 0 to 135408
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          135409 non-null  int64  
 1   topic_id            135409 non-null  int64  
 2   id                  135409 non-null  int64  
 3   parent_id           135409 non-null  int64  
 4   post_type           135409 non-null  int64  
 5   posts_count         135409 non-null  int64  
 6   view_count          135409 non-null  int64  
 7   created_at          135409 non-null  object 
 8   reply_time          135409 non-null  float64
 9   is_accepted_answer  135409 non-null  bool   
 10  author_id           135409 non-null  int64  
 11  author_username     135408 non-null  object 
dtypes: bool(1), float64(1), int64(8), object(2)
memory usage: 11.5+ MB


В каждом из датасете есть есть строчки относящиеся к пользователю system и discobot, поэтому убираем их (они являются выбросом по количеству сообщений)

In [173]:
for name, data in datas.items():
    print(f'Очищаем данные для датасета {name}...')
    data.drop(data[data['author_id'] < 0].index, inplace=True)
    print(f'Строчки с пользователями system и discobot убраны для датасета {name}\nПереводим столбец created_at в формат datetime')
    data['created_at'] = pd.to_datetime(data['created_at'], format='mixed', errors='coerce')

Очищаем данные для датасета elastic...
Строчки с пользователями system и discobot убраны для датасета elastic
Переводим столбец created_at в формат datetime
Очищаем данные для датасета freecodecamp...
Строчки с пользователями system и discobot убраны для датасета freecodecamp
Переводим столбец created_at в формат datetime
Очищаем данные для датасета python...
Строчки с пользователями system и discobot убраны для датасета python
Переводим столбец created_at в формат datetime


## Переходим к расчету наших метрик:
Для работы нам необходимо подготовить датафрейм, в котором будет выполнена агрегация по юзеру, где указано первое и последнее действие в сообществе и общее количество сообщений в течении lifetime

In [174]:
def leave_to_start_ratio_by_last_action(data):

    data = data.copy()
    
    print(f'Переводим время в формат datetime')

    data['created_at'] = pd.to_datetime(data['created_at']).dt.to_period('D')

    print(f'Выполняем агрегацию по юзеру')

    data_user_agg = data.groupby('author_id')['created_at'].agg(['min', 'max']).reset_index() #агрегируем пользователя и получаем время первого и последнего сообщения, а также общее количнство
    data_user_agg.columns = ['author_id', 'first_action', 'last_action']

    print(f'Агрегацию по пользователя прошла успешно\nПереходим к созданию временной шкалы')

    all_days = pd.period_range(
        start = data_user_agg['first_action'].min(),
        end = data_user_agg['first_action'].max(),
        freq = 'D'
    )

    print(f'Собираем информацию по ушедшим и пришедшим юзерам в течении дня')

    count_users_start = data_user_agg.groupby('first_action').size().reindex(all_days, fill_value=0) #количество новых пользователйе 
    count_users_leave = data_user_agg.groupby('last_action').size().reindex(all_days, fill_value=0) #количество ушедших пользлователей
    
    print(f'Создаем датафрейм для метрик')

    data_metrics = pd.DataFrame({
        'Новые юзеры': count_users_start,
        'Ушедшие юзеры по последнему действию': count_users_leave
    })

    print(f'Считаем коэффициент leave to start ratio')

    data_metrics['Коэффициент leave_to_start по последнему действию'] = (data_metrics['Ушедшие юзеры по последнему действию'] / data_metrics['Новые юзеры']).fillna(0)
    data_metrics['Нетто изменение по последнему действию'] = data_metrics['Новые юзеры'] - data_metrics['Ушедшие юзеры по последнему действию']

    return data_metrics

metrics_dict = {}
for name, data in datas.items():
        data_metrics_by_last_action = leave_to_start_ratio_by_last_action(data)
        metrics_dict[f'{name}_metrics_by_last_action'] = data_metrics_by_last_action
        print(f'Набор данных: {name}')
        print(data_metrics_by_last_action.head(2))
        print("-" * 60)

Переводим время в формат datetime
Выполняем агрегацию по юзеру
Агрегацию по пользователя прошла успешно
Переходим к созданию временной шкалы
Собираем информацию по ушедшим и пришедшим юзерам в течении дня
Создаем датафрейм для метрик
Считаем коэффициент leave to start ratio
Набор данных: elastic
            Новые юзеры  Ушедшие юзеры по последнему действию  \
2010-07-29            2                                     1   
2010-07-30            0                                     0   

            Коэффициент leave_to_start по последнему действию  \
2010-07-29                                                0.5   
2010-07-30                                                0.0   

            Нетто изменение по последнему действию  
2010-07-29                                       1  
2010-07-30                                       0  
------------------------------------------------------------
Переводим время в формат datetime
Выполняем агрегацию по юзеру
Агрегацию по пользователя пр

C:\Users\klyko\AppData\Local\Temp\ipykernel_18652\1199310614.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data['created_at'] = pd.to_datetime(data['created_at']).dt.to_period('D')
C:\Users\klyko\AppData\Local\Temp\ipykernel_18652\1199310614.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data['created_at'] = pd.to_datetime(data['created_at']).dt.to_period('D')
C:\Users\klyko\AppData\Local\Temp\ipykernel_18652\1199310614.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data['created_at'] = pd.to_datetime(data['created_at']).dt.to_period('D')


In [175]:
metrics_dict.keys()

dict_keys(['elastic_metrics_by_last_action', 'freecodecamp_metrics_by_last_action', 'python_metrics_by_last_action'])

In [176]:
os.makedirs("..\models", exist_ok=True)

In [177]:
#Функция будет возвращать два датафрейма
def leave_to_start_ratio_by_clv_model(data, name, b=0.5): #b - вероятность с которой считается пользователь ушедшим
    data = data.copy()

    print(f'Создаем новый датафрейм rfm сводку')

    data = data[['author_id', 'created_at']]
    

    rfm = rfm_summary(
        data,
        'author_id',
        'created_at'
    )

    rfm.to_csv(f"..\models\{name}_rfm.csv", index=False)
    
    print(f'RFM сводка успешно создана и переходим к обучению модели CLV')

    model = ParetoNBDModel(data=rfm)
    model.build_model()
    model.fit(fit_method='map')

    # Сохраняем обученную модель (InferenceData)
    model_path = f"..\models\{name}_pareto_nbd_model.nc"
    az.to_netcdf(model.idata, model_path)

    print(f'Модель обучена - считаем вероятности ухода')

    p_alive = model.expected_probability_alive()
    p_alive_clean = p_alive.squeeze()
    p_alive_values = p_alive_clean.values.ravel()

    rfm['Вероятность, что юзер живой'] = p_alive_values
    rfm['is_churn'] = rfm['Вероятность, что юзер живой'] < b

    print(f'агрегируем по пользователю и сохраняем первое и последнее время активности')
    
    data['created_at'] = pd.to_datetime(data['created_at']).dt.to_period('D')
    data_user_agg = data.groupby('author_id')['created_at'].agg(['min', 'max']).reset_index()
    data_user_agg.columns = ['customer_id', 'first_action', 'last_action']

    all_days = pd.period_range(
        start = data_user_agg['first_action'].min(),
        end = data_user_agg['first_action'].max(),
        freq = 'D'
    )

    print(f'Создаем сводную таблицу. Join по customer id')

    data_user_agg = rfm.merge(
        data_user_agg[['customer_id', 'first_action', 'last_action']],
        on = 'customer_id',
        how = 'left'
    )

    data_user_agg['Расчетное время ухода юзера (churn date)'] = np.where(
        data_user_agg['is_churn'],
        data_user_agg['last_action'],
        pd.NaT
    )

    data_user_agg['Последняя запись юзера в БД'] = data_user_agg['last_action'] #даем более читаемое название
    data_user_agg.drop(columns=['last_action'], inplace=True)
    
    churn = data_user_agg.groupby('Расчетное время ухода юзера (churn date)').size().reindex(all_days, fill_value=0)
    count_users_start = data_user_agg.groupby('first_action').size().reindex(all_days, fill_value=0) #количество новых пользователйе 
    
    data_metrics = pd.DataFrame(index=all_days)
    data_metrics['Ушедшие юзеры по модели CLV'] = churn
    data_metrics['Новые юзеры'] = count_users_start
    data_metrics['Коэффициент leave_to_start по модели CLV'] = (
        data_metrics['Ушедшие юзеры по модели CLV'] /
        data_metrics['Новые юзеры']
    ).replace([np.inf, -np.inf], np.nan).fillna(0)
    data_metrics['Нетто изменение по CLV модели'] = data_metrics['Новые юзеры'] - data_metrics['Ушедшие юзеры по модели CLV']

    data_user_agg.to_csv(f"..\models\{name}_data_user_agg.csv", index=False)

    return data_metrics

for name, data in datas.items():
        data_metrics_by_clv_model = leave_to_start_ratio_by_clv_model(data, name)
        metrics_dict[f'{name}_metrics_by_clv_model'] = data_metrics_by_clv_model #сохраняем метрики по мдели
        
        print(f'Набор данных: {name}')
        print(data_metrics_by_clv_model.head(2))
        print("-" * 60)

Создаем новый датафрейм rfm сводку


c:\Users\klyko\.conda\envs\community_management\lib\site-packages\pymc_marketing\clv\utils.py:340: UserWarning: Converting to Period representation will drop timezone information.
  .to_period(time_unit)
c:\Users\klyko\.conda\envs\community_management\lib\site-packages\pymc_marketing\clv\utils.py:226: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  transactions.set_index(datetime_col).to_period(time_unit).to_timestamp()


RFM сводка успешно создана и переходим к обучению модели CLV


Output()

Модель обучена - считаем вероятности ухода
агрегируем по пользователю и сохраняем первое и последнее время активности


...

Набор данных: freecodecamp
            Ушедшие юзеры по модели CLV  Новые юзеры  \
2016-05-12                            1            3   
2016-05-13                            0            0   

            Коэффициент leave_to_start по модели CLV  \
2016-05-12                                  0.333333   
2016-05-13                                  0.000000   

            Нетто изменение по CLV модели  
2016-05-12                              2  
2016-05-13                              0  
------------------------------------------------------------
Создаем новый датафрейм rfm сводку
RFM сводка успешно создана и переходим к обучению модели CLV


c:\Users\klyko\.conda\envs\community_management\lib\site-packages\pymc_marketing\clv\utils.py:340: UserWarning: Converting to Period representation will drop timezone information.
  .to_period(time_unit)
c:\Users\klyko\.conda\envs\community_management\lib\site-packages\pymc_marketing\clv\utils.py:226: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  transactions.set_index(datetime_col).to_period(time_unit).to_timestamp()


Output()

Модель обучена - считаем вероятности ухода
агрегируем по пользователю и сохраняем первое и последнее время активности
Создаем сводную таблицу. Join по customer id
Набор данных: python
            Ушедшие юзеры по модели CLV  Новые юзеры  \
2018-09-28                            0            7   
2018-09-29                            0           14   

            Коэффициент leave_to_start по модели CLV  \
2018-09-28                                       0.0   
2018-09-29                                       0.0   

            Нетто изменение по CLV модели  
2018-09-28                              7  
2018-09-29                             14  
------------------------------------------------------------


C:\Users\klyko\AppData\Local\Temp\ipykernel_18652\275162877.py:39: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data['created_at'] = pd.to_datetime(data['created_at']).dt.to_period('D')


In [178]:
metrics_dict.keys()

dict_keys(['elastic_metrics_by_last_action', 'freecodecamp_metrics_by_last_action', 'python_metrics_by_last_action', 'elastic_metrics_by_clv_model', 'freecodecamp_metrics_by_clv_model', 'python_metrics_by_clv_model'])

In [181]:
(pd.read_csv(f'..\models\elastic_data_user_agg.csv')).head()

,customer_id,frequency,recency,T,"Вероятность, что юзер живой",is_churn,first_action,Расчетное время ухода юзера (churn date),Последняя запись юзера в БД
0,2,16.0,716.0,3542.0,4.246763e-11,True,2015-06-15,2017-05-31,2017-05-31
1,5,45.0,3303.0,3584.0,6.616259e-01,False,2015-05-04,NaN,2024-05-19
2,7,12.0,485.0,3500.0,1.389963e-10,True,2015-07-27,2016-11-23,2016-11-23
3,8,1.0,90.0,3532.0,1.432838e-03,True,2015-06-25,2015-09-23,2015-09-23
4,10,7.0,56.0,3588.0,9.019784e-14,True,2015-04-30,2015-06-25,2015-06-25


In [184]:

def predict_churn_date(name, b=0.5):

    print(f'Загружаем модель и данные для набора {name}')
    idata = az.from_netcdf(f"..\models\{name}_pareto_nbd_model.nc")
    rfm = pd.read_csv(f'..\models\{name}_rfm.csv')
    model = ParetoNBDModel(data=rfm)
    model.build_model()
    model.idata = idata

    print(f'Оставляем тех, для кого на последнюю запись в БД модель выдает вероятность быть живым больше {b}')

    data_user_agg = pd.read_csv(f"..\models\{name}_data_user_agg.csv")
    data_user_agg['Последняя запись юзера в БД'] = pd.to_datetime(
         data_user_agg['Последняя запись юзера в БД'], format = 'mixed', errors='coerce')
    churn_users = data_user_agg.loc[data_user_agg['is_churn'], 'customer_id'].tolist()
    data_user_agg_predict = data_user_agg[~data_user_agg['customer_id'].isin(churn_users)].copy()

    active_users = data_user_agg_predict['customer_id'].tolist()

    days_forward = 0
    horizon = 365

    while len(active_users) > 0 and days_forward < horizon:
        days_forward += 1
        print(f"День {days_forward}: осталось пользователей {len(active_users)}")

        # Увеличиваем время наблюдения Т на 1 день
        data_user_agg_predict.loc[data_user_agg_predict['customer_id'].isin(active_users), 'T'] += 1

        #Рассчитываем вероятность быть живым для активных пользователей

        p_alive = model.expected_probability_alive(data_user_agg_predict)
        p_alive_clean = p_alive.squeeze()
        p_alive_values = p_alive_clean.values.ravel()
        data_user_agg_predict['Вероятность, что юзер живой'] = p_alive_values

        churn_that_day = data_user_agg_predict[
            (data_user_agg_predict['customer_id'].isin(active_users)) &
            (data_user_agg_predict['Вероятность, что юзер живой'] < b)
        ]

        mask = data_user_agg_predict['customer_id'].isin(churn_that_day['customer_id'])
        data_user_agg_predict.loc[
            mask,
            'Расчетное время ухода юзера (churn date)'
        ] = (
            data_user_agg_predict.loc[mask, 'Последняя запись юзера в БД']
            + pd.to_timedelta(days_forward, unit='D')
        )
        
        active_users = list(
            set(active_users) -
            set(churn_that_day['customer_id'])
        )

    data_user_agg = data_user_agg.merge(
        data_user_agg_predict[['customer_id', 'Расчетное время ухода юзера (churn date)']],
        on='customer_id',
        how='left'
    )

    return data_user_agg

for name, data in datas.items():
        data_user_agg_predicted = predict_churn_date(name)
        metrics_dict[f'{name}_lifetime_predicted'] = data_user_agg_predicted #сохраняем метрики по мдели
        
        print(f'Набор данных: {name}')
        print(data_user_agg_predicted.head(2))
        print("-" * 60)

Загружаем модель и данные для набора elastic
Оставляем тех, для кого на последнюю запись в БД модель выдает вероятность быть живым больше 0.5
День 1: осталось пользователей 1468
День 2: осталось пользователей 1451
День 3: осталось пользователей 1435
День 4: осталось пользователей 1426
День 5: осталось пользователей 1417
День 6: осталось пользователей 1404
День 7: осталось пользователей 1390
День 8: осталось пользователей 1373
День 9: осталось пользователей 1362
День 10: осталось пользователей 1353
День 11: осталось пользователей 1345
День 12: осталось пользователей 1328
День 13: осталось пользователей 1303
День 14: осталось пользователей 1298
День 15: осталось пользователей 1293
День 16: осталось пользователей 1289
День 17: осталось пользователей 1278
День 18: осталось пользователей 1276
День 19: осталось пользователей 1270
День 20: осталось пользователей 1264
День 21: осталось пользователей 1261
День 22: осталось пользователей 1246
День 23: осталось пользователей 1239
День 24: осталос

KeyboardInterrupt: 